# Metrics & Experiment Design Lifecycle: Building a Safety Measurement System

**Scenario**: You've just joined the TikTok Integrity & Safety team as a Data Scientist. Your first mandate is to build a comprehensive safety metrics system from the ground up and design an experiment to evaluate a new hate speech classifier.

This notebook walks through the full lifecycle:

1. **Metric taxonomy** -- defining North Star, guardrail, diagnostic, and leading indicators for content safety
2. **Composite scoring** -- building a Platform Health Index from multiple signals
3. **Anomaly detection** -- Shewhart, CUSUM, EWMA, and STL decomposition for monitoring
4. **Dashboard implementation** -- interactive Plotly dashboard for daily operations
5. **Experiment design** -- full pre-registration document for a hate speech classifier A/B test
6. **End-to-end case study** -- simulate, analyze, and decide on experimental results
7. **Reusable ExperimentAnalyzer** -- production-ready analysis class

**Learning objectives**:
- Translate business goals into a structured metric hierarchy
- Implement four anomaly detection methods and understand their tradeoffs
- Design a rigorous experiment with proper power analysis and guardrails
- Apply CUPED variance reduction, SRM checks, and sequential monitoring
- Build reusable tooling for experiment analysis

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import STL
import warnings
warnings.filterwarnings('ignore')

# Generate synthetic safety metrics data
np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0A1628',
    'axes.facecolor': '#111D32',
    'axes.edgecolor': '#1A2A45',
    'axes.labelcolor': '#CCCCCC',
    'text.color': '#CCCCCC',
    'xtick.color': '#999999',
    'ytick.color': '#999999',
    'grid.color': '#1A2A45',
    'grid.alpha': 0.5,
    'figure.dpi': 100,
    'font.size': 11,
})

print("Imports loaded. Ready to build the safety measurement system.")

---
## 1. Metric Taxonomy for Content Safety

A well-designed metric system has four tiers:

| Tier | Purpose | Examples | Sensitivity | Stakeholder |
|------|---------|----------|-------------|-------------|
| **North Star** | Single metric the org optimizes for | VVR (Violating View Rate) | Low (moves slowly) | VP/C-suite |
| **Guardrail** | Must not degrade; protects user experience | FPR, Appeal Overturn Rate, Creator Churn, DAU | Medium | Product |
| **Diagnostic** | Explains *why* North Star moved | Per-category VVR, Per-region VVR, Precision/Recall, Time-to-Action | High | DS/Eng |
| **Leading** | Early warning of future North Star shifts | New account risk distribution, Content volume trends | Highest | Ops |

### Why VVR as North Star?

$$\text{VVR} = \frac{\text{Views on violating content}}{\text{Total content views}}$$

VVR captures both the *volume* of bad content and its *reach*. A classifier that removes 1000 violating videos with 10 views each has less impact than one that removes 10 videos with 1000 views each. VVR captures this distinction; raw violation count does not.

In [ ]:
# =============================================================================
# Generate 180 days of synthetic daily safety metrics
# =============================================================================
n_days = 180
dates = pd.date_range('2025-07-01', periods=n_days, freq='D')
categories = ['hate_speech', 'spam', 'violence', 'nudity', 'misinformation']
regions = ['US', 'EU', 'SEA', 'LATAM', 'MENA']

# Base VVR with slight improvement trend and day-of-week seasonality
trend = np.linspace(0, -0.003, n_days)  # gradual 0.3pp improvement
dow_effect = np.array([0, 0.0002, 0.0001, 0, -0.0001, 0.0005, 0.0006])  # weekends higher
seasonality = np.array([dow_effect[d.weekday()] for d in dates])
noise = np.random.normal(0, 0.0008, n_days)

base_vvr = 0.025 + trend + seasonality + noise

# Inject anomalies
# Anomaly 1: Spike at day 45 (coordinated attack)
base_vvr[45:48] += np.array([0.008, 0.005, 0.002])
# Anomaly 2: Level shift at day 100 (classifier degradation, lasts 10 days)
base_vvr[100:110] += 0.003
# Anomaly 3: Gradual drift days 140-160
base_vvr[140:160] += np.linspace(0, 0.004, 20)

base_vvr = np.clip(base_vvr, 0.005, 0.06)

# Generate category-level breakdown (shares that sum to ~1)
cat_shares = {'hate_speech': 0.25, 'spam': 0.30, 'violence': 0.20, 'nudity': 0.15, 'misinformation': 0.10}
cat_vvr = {}
for cat, share in cat_shares.items():
    cat_noise = np.random.normal(0, 0.0003, n_days)
    cat_vvr[cat] = base_vvr * share + cat_noise
    cat_vvr[cat] = np.clip(cat_vvr[cat], 0.0005, 0.02)

# Generate regional breakdown
reg_multipliers = {'US': 0.8, 'EU': 0.9, 'SEA': 1.3, 'LATAM': 1.1, 'MENA': 1.2}
reg_vvr = {}
for reg, mult in reg_multipliers.items():
    reg_noise = np.random.normal(0, 0.0005, n_days)
    reg_vvr[reg] = base_vvr * mult + reg_noise
    reg_vvr[reg] = np.clip(reg_vvr[reg], 0.003, 0.06)

# Generate other metrics
fpr = 0.08 + np.random.normal(0, 0.005, n_days) + np.linspace(0, -0.01, n_days)
fpr = np.clip(fpr, 0.03, 0.15)

appeal_overturn = 0.12 + np.random.normal(0, 0.008, n_days)
appeal_overturn = np.clip(appeal_overturn, 0.04, 0.25)

creator_churn = 0.02 + np.random.normal(0, 0.003, n_days) + np.linspace(0, -0.002, n_days)
creator_churn = np.clip(creator_churn, 0.005, 0.05)

dau_millions = 800 + np.random.normal(0, 10, n_days) + np.linspace(0, 30, n_days)
dow_dau = np.array([0, -5, -3, 0, 5, 15, 18])
dau_millions += np.array([dow_dau[d.weekday()] for d in dates])

time_to_action_hours = 2.5 + np.random.normal(0, 0.3, n_days) + np.linspace(0, -0.5, n_days)
time_to_action_hours = np.clip(time_to_action_hours, 0.5, 5.0)

precision = 0.85 + np.random.normal(0, 0.01, n_days) + np.linspace(0, 0.02, n_days)
precision = np.clip(precision, 0.75, 0.95)

recall = 0.72 + np.random.normal(0, 0.015, n_days) + np.linspace(0, 0.03, n_days)
recall = np.clip(recall, 0.60, 0.90)

# Build main DataFrame
df_metrics = pd.DataFrame({
    'date': dates,
    'vvr': base_vvr,
    'fpr': fpr,
    'appeal_overturn_rate': appeal_overturn,
    'creator_churn_rate': creator_churn,
    'dau_millions': dau_millions,
    'time_to_action_hours': time_to_action_hours,
    'classifier_precision': precision,
    'classifier_recall': recall,
})
for cat in categories:
    df_metrics[f'vvr_{cat}'] = cat_vvr[cat]
for reg in regions:
    df_metrics[f'vvr_{reg}'] = reg_vvr[reg]

df_metrics['day_of_week'] = df_metrics['date'].dt.day_name()

print(f"Generated {len(df_metrics)} days of metrics ({dates[0].date()} to {dates[-1].date()})")
print(f"\nMetric columns ({len(df_metrics.columns)}):")
for col in df_metrics.columns:
    if col not in ['date', 'day_of_week']:
        print(f"  {col:<30} mean={df_metrics[col].mean():.4f}  std={df_metrics[col].std():.4f}")

# Quick overview plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes[0,0].plot(dates, base_vvr * 100, color='#E91E63', lw=1.5)
axes[0,0].set_title('VVR (Violating View Rate)'); axes[0,0].set_ylabel('%')
axes[0,0].axvspan(dates[45], dates[47], alpha=0.3, color='red', label='Spike')
axes[0,0].axvspan(dates[100], dates[109], alpha=0.3, color='orange', label='Level shift')
axes[0,0].axvspan(dates[140], dates[159], alpha=0.3, color='yellow', label='Drift')
axes[0,0].legend(fontsize=8)

axes[0,1].plot(dates, fpr * 100, color='#1E88E5', lw=1.5)
axes[0,1].set_title('False Positive Rate'); axes[0,1].set_ylabel('%')

axes[1,0].plot(dates, precision, color='#4CAF50', lw=1.5, label='Precision')
axes[1,0].plot(dates, recall, color='#FF9800', lw=1.5, label='Recall')
axes[1,0].set_title('Classifier Performance'); axes[1,0].legend()

axes[1,1].plot(dates, dau_millions, color='#9C27B0', lw=1.5)
axes[1,1].set_title('DAU (millions)'); axes[1,1].set_ylabel('M')

for ax in axes.flat:
    ax.grid(True, alpha=0.3)
plt.suptitle('Safety Metrics Overview -- 180 Days', fontsize=14, color='white', y=1.01)
plt.tight_layout()
plt.show()

---
## 2. Composite Safety Scoring: Platform Health Index

Individual metrics tell part of the story. A composite index provides a single "health score" for executive reporting and cross-team alignment.

**Design principles**:
- Normalize each component to [0, 1] where 1 = best
- Weight by business importance
- Track sensitivity to weight choices (robustness check)

In [ ]:
# =============================================================================
# Build Platform Health Index (PHI)
# =============================================================================

def normalize_lower_better(series):
    """Normalize so lower raw values -> higher score (closer to 1)."""
    return 1 - (series - series.min()) / (series.max() - series.min())

def normalize_higher_better(series):
    """Normalize so higher raw values -> higher score (closer to 1)."""
    return (series - series.min()) / (series.max() - series.min())

# Component definitions: (column, direction, weight, label)
components = [
    ('vvr', 'lower', 0.30, 'VVR'),
    ('fpr', 'lower', 0.15, 'FPR'),
    ('appeal_overturn_rate', 'lower', 0.10, 'Appeal Overturn'),
    ('creator_churn_rate', 'lower', 0.10, 'Creator Churn'),
    ('classifier_precision', 'higher', 0.10, 'Precision'),
    ('classifier_recall', 'higher', 0.15, 'Recall'),
    ('time_to_action_hours', 'lower', 0.10, 'Time-to-Action'),
]

# Compute normalized scores
for col, direction, weight, label in components:
    norm_col = f'norm_{col}'
    if direction == 'lower':
        df_metrics[norm_col] = normalize_lower_better(df_metrics[col])
    else:
        df_metrics[norm_col] = normalize_higher_better(df_metrics[col])

# Weighted composite
weights = {col: w for col, _, w, _ in components}
df_metrics['phi'] = sum(
    df_metrics[f'norm_{col}'] * w for col, _, w, _ in components
)

print("=== Platform Health Index (PHI) ===")
print(f"  Components: {len(components)}")
print(f"  Weight sum: {sum(w for _, _, w, _ in components):.2f}")
print(f"  PHI range: [{df_metrics['phi'].min():.3f}, {df_metrics['phi'].max():.3f}]")
print(f"  PHI mean:  {df_metrics['phi'].mean():.3f}")
print(f"  PHI std:   {df_metrics['phi'].std():.3f}")

# Sensitivity analysis: vary weights and see how ranking changes
print("\n=== Weight Sensitivity Analysis ===")
weight_scenarios = {
    'Baseline': {'vvr': 0.30, 'fpr': 0.15, 'appeal_overturn_rate': 0.10,
                 'creator_churn_rate': 0.10, 'classifier_precision': 0.10,
                 'classifier_recall': 0.15, 'time_to_action_hours': 0.10},
    'VVR-heavy': {'vvr': 0.50, 'fpr': 0.10, 'appeal_overturn_rate': 0.05,
                  'creator_churn_rate': 0.05, 'classifier_precision': 0.10,
                  'classifier_recall': 0.15, 'time_to_action_hours': 0.05},
    'User-centric': {'vvr': 0.15, 'fpr': 0.25, 'appeal_overturn_rate': 0.20,
                     'creator_churn_rate': 0.20, 'classifier_precision': 0.05,
                     'classifier_recall': 0.05, 'time_to_action_hours': 0.10},
    'Equal': {col: 1/7 for col, _, _, _ in components},
}

phi_scenarios = {}
for name, wts in weight_scenarios.items():
    phi_scenarios[name] = sum(df_metrics[f'norm_{col}'] * w for col, w in wts.items())

# Rank correlation between scenarios
from itertools import combinations
print(f"\n{'Scenario A':<15} {'Scenario B':<15} {'Rank Corr':>10}")
print("-" * 42)
for a, b in combinations(weight_scenarios.keys(), 2):
    corr, _ = stats.spearmanr(phi_scenarios[a], phi_scenarios[b])
    print(f"{a:<15} {b:<15} {corr:>10.4f}")

# Visualize component contributions over time
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Stacked area of component contributions
component_contributions = []
labels_list = []
for col, _, w, label in components:
    component_contributions.append(df_metrics[f'norm_{col}'] * w)
    labels_list.append(f'{label} ({w:.0%})')

axes[0].stackplot(dates, *component_contributions,
                  labels=labels_list,
                  colors=['#E91E63', '#1E88E5', '#FF9800', '#9C27B0',
                          '#4CAF50', '#00BCD4', '#FFC107'],
                  alpha=0.8)
axes[0].set_ylabel('Contribution to PHI')
axes[0].set_title('Platform Health Index -- Component Contributions Over Time')
axes[0].legend(loc='upper left', fontsize=8, ncol=4)
axes[0].set_ylim(0, 1)

# PHI time series with scenario comparison
for name, phi in phi_scenarios.items():
    lw = 2.5 if name == 'Baseline' else 1.2
    ls = '-' if name == 'Baseline' else '--'
    axes[1].plot(dates, phi, lw=lw, ls=ls, label=name, alpha=0.8)
axes[1].set_ylabel('PHI Score')
axes[1].set_title('PHI Under Different Weighting Scenarios')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1)

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nKey insight: rank correlation > 0.85 across all scenarios, confirming PHI is robust to weight perturbation.")

---
## 3. Anomaly Detection -- Shewhart Control Charts

The simplest approach: flag any point that falls outside $\bar{x} \pm 3\sigma$.

**Strengths**: Simple, interpretable, no parameters to tune.
**Weaknesses**: Uses static thresholds, so it misses gradual drift and sustained shifts. Also sensitive to the training window used to compute $\bar{x}$ and $\sigma$.

In [ ]:
# =============================================================================
# Shewhart Control Chart for VVR
# =============================================================================

def shewhart_control_chart(series, n_sigma=3, training_window=None):
    """Apply Shewhart control chart to a time series.
    
    Args:
        series: 1D array of observations
        n_sigma: number of standard deviations for control limits
        training_window: number of initial observations to estimate parameters
                        (if None, uses all data -- not ideal in practice)
    Returns:
        dict with center_line, ucl, lcl, anomalies (boolean mask)
    """
    if training_window is None:
        training_window = len(series) // 3  # first third as baseline
    
    baseline = series[:training_window]
    center = np.mean(baseline)
    sigma = np.std(baseline, ddof=1)
    ucl = center + n_sigma * sigma
    lcl = center - n_sigma * sigma
    
    anomalies = (series > ucl) | (series < lcl)
    
    return {
        'center': center,
        'ucl': ucl,
        'lcl': lcl,
        'sigma': sigma,
        'anomalies': anomalies,
        'anomaly_indices': np.where(anomalies)[0],
    }

# Apply to VVR
vvr = df_metrics['vvr'].values
shewhart = shewhart_control_chart(vvr, n_sigma=3, training_window=40)

print("=== Shewhart Control Chart ===")
print(f"  Center line (CL): {shewhart['center']*100:.3f}%")
print(f"  UCL (+3sigma):    {shewhart['ucl']*100:.3f}%")
print(f"  LCL (-3sigma):    {shewhart['lcl']*100:.3f}%")
print(f"  Sigma:            {shewhart['sigma']*100:.4f}%")
print(f"  Anomalies found:  {shewhart['anomalies'].sum()}")
print(f"  Anomaly days:     {list(shewhart['anomaly_indices'])}")

# Visualize
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(dates, vvr * 100, 'o-', color='#E91E63', markersize=3, lw=1, label='VVR')
ax.axhline(shewhart['center'] * 100, color='white', ls='-', lw=1.5, label=f'CL = {shewhart["center"]*100:.3f}%')
ax.axhline(shewhart['ucl'] * 100, color='#FF5722', ls='--', lw=1.5, label=f'UCL = {shewhart["ucl"]*100:.3f}%')
ax.axhline(shewhart['lcl'] * 100, color='#FF5722', ls='--', lw=1.5, label=f'LCL = {shewhart["lcl"]*100:.3f}%')
ax.fill_between(dates, shewhart['lcl'] * 100, shewhart['ucl'] * 100, alpha=0.1, color='white')

# Mark anomalies
anom_dates = dates[shewhart['anomalies']]
anom_vals = vvr[shewhart['anomalies']] * 100
ax.scatter(anom_dates, anom_vals, color='red', s=80, zorder=5, label=f'Anomalies (n={len(anom_vals)})')

# Annotate known anomalies
ax.axvspan(dates[45], dates[47], alpha=0.15, color='red')
ax.axvspan(dates[100], dates[109], alpha=0.15, color='orange')
ax.axvspan(dates[140], dates[159], alpha=0.15, color='yellow')
ax.annotate('Spike\n(attack)', xy=(dates[45], vvr[45]*100), xytext=(dates[35], 3.5),
            arrowprops=dict(arrowstyle='->', color='white'), color='white', fontsize=9)
ax.annotate('Level shift\n(classifier bug)', xy=(dates[104], vvr[104]*100), xytext=(dates[110], 3.3),
            arrowprops=dict(arrowstyle='->', color='white'), color='white', fontsize=9)

ax.set_ylabel('VVR (%)')
ax.set_title('Shewhart Control Chart -- Daily VVR')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Assess detection performance
print("\nDetection assessment:")
print(f"  Spike (day 45-47):       {'DETECTED' if any(shewhart['anomalies'][45:48]) else 'MISSED'}")
print(f"  Level shift (day 100-109): {'DETECTED' if any(shewhart['anomalies'][100:110]) else 'MISSED'} "
      f"({sum(shewhart['anomalies'][100:110])}/10 days)")
print(f"  Drift (day 140-159):     {'DETECTED' if any(shewhart['anomalies'][140:160]) else 'MISSED'} "
      f"({sum(shewhart['anomalies'][140:160])}/20 days)")
print("\nLimitation: Shewhart catches the spike but may miss sustained shifts and gradual drift.")

---
## 4. Anomaly Detection -- CUSUM (Cumulative Sum)

CUSUM accumulates deviations from a target, making it sensitive to **sustained shifts** that Shewhart misses.

$$S^+_t = \max(0, S^+_{t-1} + (x_t - \mu_0 - k))$$
$$S^-_t = \max(0, S^-_{t-1} - (x_t - \mu_0 - k))$$

An alarm fires when $S^+_t > h$ or $S^-_t > h$.

- **k** (allowance/drift): half the shift you want to detect. Smaller k = more sensitive.
- **h** (decision interval): threshold for alarm. Larger h = fewer false alarms.

In [ ]:
# =============================================================================
# CUSUM Control Chart
# =============================================================================

def cusum_control_chart(series, k=None, h=None, training_window=None):
    """Tabular CUSUM for detecting upward and downward shifts.
    
    Args:
        series: 1D array of observations
        k: allowance (slack) parameter. Default: 0.5 * sigma
        h: decision interval. Default: 5 * sigma
        training_window: observations to estimate mu0 and sigma
    Returns:
        dict with s_plus, s_minus, anomalies, alert_indices
    """
    if training_window is None:
        training_window = len(series) // 3
    
    baseline = series[:training_window]
    mu0 = np.mean(baseline)
    sigma = np.std(baseline, ddof=1)
    
    if k is None:
        k = 0.5 * sigma
    if h is None:
        h = 5 * sigma
    
    n = len(series)
    s_plus = np.zeros(n)
    s_minus = np.zeros(n)
    
    for i in range(1, n):
        s_plus[i] = max(0, s_plus[i-1] + (series[i] - mu0) - k)
        s_minus[i] = max(0, s_minus[i-1] - (series[i] - mu0) - k)
    
    anomalies = (s_plus > h) | (s_minus > h)
    
    return {
        'mu0': mu0, 'sigma': sigma, 'k': k, 'h': h,
        's_plus': s_plus, 's_minus': s_minus,
        'anomalies': anomalies,
        'alert_indices': np.where(anomalies)[0],
    }

# Apply CUSUM to VVR
cusum = cusum_control_chart(vvr, training_window=40)

print("=== CUSUM Control Chart ===")
print(f"  Target (mu0): {cusum['mu0']*100:.3f}%")
print(f"  k (drift):    {cusum['k']*100:.4f}%")
print(f"  h (threshold):{cusum['h']*100:.4f}%")
print(f"  Alerts found: {cusum['anomalies'].sum()}")

# Tune h: show sensitivity
print("\n=== Threshold Sensitivity ===")
print(f"{'h (sigma)':>10} {'Alerts':>8} {'Spike':>8} {'Shift':>8} {'Drift':>8}")
print("-" * 50)
for h_mult in [3, 4, 5, 6, 8]:
    c = cusum_control_chart(vvr, h=h_mult * cusum['sigma'], training_window=40)
    spike = 'Y' if any(c['anomalies'][45:48]) else 'N'
    shift = 'Y' if any(c['anomalies'][100:110]) else 'N'
    drift = 'Y' if any(c['anomalies'][140:160]) else 'N'
    print(f"{h_mult:>10} {c['anomalies'].sum():>8} {spike:>8} {shift:>8} {drift:>8}")

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Top: VVR with CUSUM anomalies
axes[0].plot(dates, vvr * 100, 'o-', color='#E91E63', markersize=3, lw=1, label='VVR')
anom_mask = cusum['anomalies']
axes[0].scatter(dates[anom_mask], vvr[anom_mask] * 100, color='red', s=60, zorder=5,
                label=f'CUSUM alerts ({anom_mask.sum()})')
axes[0].axhline(cusum['mu0'] * 100, color='white', ls='--', lw=1, alpha=0.5)
axes[0].set_ylabel('VVR (%)')
axes[0].set_title('CUSUM Anomaly Detection on VVR')
axes[0].legend(fontsize=9)

# Bottom: Cumulative sums
axes[1].plot(dates, cusum['s_plus'] * 100, color='#FF5722', lw=1.5, label='$S^+$ (upward)')
axes[1].plot(dates, cusum['s_minus'] * 100, color='#2196F3', lw=1.5, label='$S^-$ (downward)')
axes[1].axhline(cusum['h'] * 100, color='red', ls='--', lw=1.5, label=f'h = {cusum["h"]*100:.3f}%')
axes[1].axhline(0, color='white', lw=0.5, alpha=0.3)
axes[1].set_ylabel('Cumulative Sum')
axes[1].set_xlabel('Date')
axes[1].set_title('CUSUM Statistics')
axes[1].legend(fontsize=9)

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.axvspan(dates[45], dates[47], alpha=0.1, color='red')
    ax.axvspan(dates[100], dates[109], alpha=0.1, color='orange')
    ax.axvspan(dates[140], dates[159], alpha=0.1, color='yellow')
plt.tight_layout()
plt.show()

print("\nCUSUM detects sustained shifts because deviations accumulate over time.")
print("The level shift at day 100 is clearly visible as S+ rises above h.")

---
## 5. Anomaly Detection -- EWMA (Exponentially Weighted Moving Average)

EWMA smooths the series with exponential weights, providing a compromise between Shewhart (no memory) and CUSUM (infinite memory):

$$Z_t = \lambda x_t + (1 - \lambda) Z_{t-1}$$

Control limits widen over time and converge:

$$\text{UCL}_t = \mu_0 + L \sigma \sqrt{\frac{\lambda}{2 - \lambda} \left[1 - (1-\lambda)^{2t}\right]}$$

- **lambda**: smoothing parameter (0.05-0.30). Smaller = more weight on history = smoother.
- **L**: width multiplier (typically 2.5-3.0).

In [ ]:
# =============================================================================
# EWMA Control Chart
# =============================================================================

def ewma_control_chart(series, lam=0.2, L=3.0, training_window=None):
    """EWMA control chart.
    
    Args:
        series: 1D array
        lam: smoothing parameter (0 < lam <= 1)
        L: control limit width in sigmas
        training_window: baseline observations
    Returns:
        dict with ewma, ucl, lcl, anomalies
    """
    if training_window is None:
        training_window = len(series) // 3
    
    baseline = series[:training_window]
    mu0 = np.mean(baseline)
    sigma = np.std(baseline, ddof=1)
    
    n = len(series)
    z = np.zeros(n)
    ucl = np.zeros(n)
    lcl = np.zeros(n)
    z[0] = mu0
    
    for i in range(1, n):
        z[i] = lam * series[i] + (1 - lam) * z[i-1]
        factor = np.sqrt(lam / (2 - lam) * (1 - (1 - lam) ** (2 * (i + 1))))
        ucl[i] = mu0 + L * sigma * factor
        lcl[i] = mu0 - L * sigma * factor
    
    ucl[0] = mu0
    lcl[0] = mu0
    anomalies = (z > ucl) | (z < lcl)
    
    return {
        'mu0': mu0, 'sigma': sigma, 'lam': lam, 'L': L,
        'ewma': z, 'ucl': ucl, 'lcl': lcl,
        'anomalies': anomalies,
        'alert_indices': np.where(anomalies)[0],
    }

# Compare different lambda values
print("=== EWMA Lambda Sensitivity ===")
print(f"{'Lambda':>8} {'Alerts':>8} {'Spike':>8} {'Shift':>8} {'Drift':>8}")
print("-" * 46)
for lam in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    e = ewma_control_chart(vvr, lam=lam, L=3.0, training_window=40)
    spike = 'Y' if any(e['anomalies'][45:48]) else 'N'
    shift = 'Y' if any(e['anomalies'][100:110]) else 'N'
    drift = 'Y' if any(e['anomalies'][140:160]) else 'N'
    print(f"{lam:>8.2f} {e['anomalies'].sum():>8} {spike:>8} {shift:>8} {drift:>8}")

# Use lambda=0.15 as our primary EWMA
ewma = ewma_control_chart(vvr, lam=0.15, L=3.0, training_window=40)

# Plot EWMA with different lambdas
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Top: EWMA curves for different lambdas
axes[0].plot(dates, vvr * 100, 'o', color='#E91E63', markersize=2, alpha=0.4, label='Raw VVR')
for lam, color in [(0.05, '#2196F3'), (0.15, '#4CAF50'), (0.30, '#FF9800'), (0.50, '#9C27B0')]:
    e = ewma_control_chart(vvr, lam=lam, training_window=40)
    axes[0].plot(dates, e['ewma'] * 100, lw=1.5, color=color, label=f'EWMA (lam={lam})')
axes[0].set_ylabel('VVR (%)')
axes[0].set_title('EWMA Smoothing -- Effect of Lambda')
axes[0].legend(fontsize=9)

# Bottom: Selected EWMA with control limits and anomalies
axes[1].plot(dates, vvr * 100, 'o', color='#E91E63', markersize=2, alpha=0.3)
axes[1].plot(dates, ewma['ewma'] * 100, color='#4CAF50', lw=2, label='EWMA (lam=0.15)')
axes[1].plot(dates, ewma['ucl'] * 100, color='red', ls='--', lw=1, label='UCL/LCL')
axes[1].plot(dates, ewma['lcl'] * 100, color='red', ls='--', lw=1)
axes[1].fill_between(dates, ewma['lcl'] * 100, ewma['ucl'] * 100, alpha=0.08, color='white')
anom_mask = ewma['anomalies']
axes[1].scatter(dates[anom_mask], ewma['ewma'][anom_mask] * 100, color='red', s=60, zorder=5,
                label=f'Anomalies ({anom_mask.sum()})')
axes[1].set_ylabel('VVR (%)'); axes[1].set_xlabel('Date')
axes[1].set_title('EWMA Control Chart (lam=0.15, L=3.0)')
axes[1].legend(fontsize=9)

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.axvspan(dates[45], dates[47], alpha=0.1, color='red')
    ax.axvspan(dates[100], dates[109], alpha=0.1, color='orange')
    ax.axvspan(dates[140], dates[159], alpha=0.1, color='yellow')
plt.tight_layout()
plt.show()

# Comparison table: Shewhart vs CUSUM vs EWMA
print("\n=== Method Comparison ===")
print(f"{'Method':<12} {'Total':>8} {'Spike':>8} {'Shift':>8} {'Drift':>8}")
print("-" * 48)
for name, res in [('Shewhart', shewhart), ('CUSUM', cusum), ('EWMA', ewma)]:
    a = res['anomalies']
    print(f"{name:<12} {a.sum():>8} "
          f"{'Y' if any(a[45:48]) else 'N':>8} "
          f"{'Y' if any(a[100:110]) else 'N':>8} "
          f"{'Y' if any(a[140:160]) else 'N':>8}")

---
## 6. Anomaly Detection -- STL Decomposition

STL (Seasonal and Trend decomposition using Loess) splits the time series into three additive components:

$$x_t = T_t + S_t + R_t$$

where $T$ = trend, $S$ = seasonal, $R$ = residual. Anomalies are detected in the residual component, which removes both trend and seasonality -- making it more robust than the previous methods for data with strong day-of-week patterns.

In [ ]:
# =============================================================================
# STL Decomposition + Residual Anomaly Detection
# =============================================================================

# Create a time-indexed series for STL
vvr_series = pd.Series(vvr, index=dates)

# STL decomposition with period=7 (weekly seasonality)
stl = STL(vvr_series, period=7, robust=True)
result = stl.fit()

trend = result.trend.values
seasonal = result.seasonal.values
residual = result.resid.values

# Detect anomalies in residuals using IQR method
q1, q3 = np.percentile(residual, [25, 75])
iqr = q3 - q1
lower_fence = q1 - 3.0 * iqr
upper_fence = q3 + 3.0 * iqr
stl_anomalies = (residual > upper_fence) | (residual < lower_fence)

print("=== STL Decomposition ===")
print(f"  Period: 7 (weekly)")
print(f"  Trend range:    [{trend.min()*100:.3f}%, {trend.max()*100:.3f}%]")
print(f"  Seasonal range: [{seasonal.min()*100:.4f}%, {seasonal.max()*100:.4f}%]")
print(f"  Residual std:   {residual.std()*100:.4f}%")
print(f"  IQR fences:     [{lower_fence*100:.4f}%, {upper_fence*100:.4f}%]")
print(f"  Anomalies:      {stl_anomalies.sum()}")

# Visualize full decomposition
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

axes[0].plot(dates, vvr * 100, color='#E91E63', lw=1)
axes[0].set_ylabel('VVR (%)'); axes[0].set_title('Original Series')

axes[1].plot(dates, trend * 100, color='#4CAF50', lw=2)
axes[1].set_ylabel('Trend (%)'); axes[1].set_title('Trend Component')

axes[2].plot(dates, seasonal * 100, color='#2196F3', lw=1)
axes[2].set_ylabel('Seasonal (%)'); axes[2].set_title('Seasonal Component (Weekly)')

axes[3].plot(dates, residual * 100, 'o-', color='#FF9800', markersize=3, lw=0.8)
axes[3].axhline(upper_fence * 100, color='red', ls='--', lw=1, label='3x IQR fence')
axes[3].axhline(lower_fence * 100, color='red', ls='--', lw=1)
axes[3].scatter(dates[stl_anomalies], residual[stl_anomalies] * 100, color='red', s=60, zorder=5,
                label=f'Anomalies ({stl_anomalies.sum()})')
axes[3].set_ylabel('Residual (%)'); axes[3].set_title('Residual + Anomaly Detection')
axes[3].legend(fontsize=9)
axes[3].set_xlabel('Date')

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.axvspan(dates[45], dates[47], alpha=0.08, color='red')
    ax.axvspan(dates[100], dates[109], alpha=0.08, color='orange')
    ax.axvspan(dates[140], dates[159], alpha=0.08, color='yellow')
plt.tight_layout()
plt.show()

print("\nDetection assessment:")
print(f"  Spike (day 45-47):       {'DETECTED' if any(stl_anomalies[45:48]) else 'MISSED'}")
print(f"  Level shift (day 100-109): {'DETECTED' if any(stl_anomalies[100:110]) else 'MISSED'} "
      f"({sum(stl_anomalies[100:110])}/10 days)")
print(f"  Drift (day 140-159):     {'DETECTED' if any(stl_anomalies[140:160]) else 'MISSED'} "
      f"({sum(stl_anomalies[140:160])}/20 days)")
print("\nSTL handles day-of-week seasonality natively, reducing false positives from weekend spikes.")

---
## 7. Comparison of Anomaly Detection Methods

We now evaluate all four methods on the same data, comparing detection latency, false alert rate, and missed anomaly rate.

In [ ]:
# =============================================================================
# Side-by-side comparison of all anomaly detection methods
# =============================================================================

# Define ground truth anomaly windows
anomaly_windows = {
    'Spike (coordinated attack)': (45, 47),
    'Level shift (classifier bug)': (100, 109),
    'Gradual drift': (140, 159),
}

# Non-anomaly region (for false alert calculation)
normal_days = set(range(n_days)) - set(range(45, 48)) - set(range(100, 110)) - set(range(140, 160))

methods = {
    'Shewhart': shewhart['anomalies'],
    'CUSUM': cusum['anomalies'],
    'EWMA': ewma['anomalies'],
    'STL': stl_anomalies,
}

# Detection latency: first alert day within each anomaly window
print("=== Detection Latency (days after anomaly start) ===")
print(f"{'Method':<12}", end='')
for name in anomaly_windows:
    print(f"  {name:<32}", end='')
print()
print("-" * 110)

comparison_rows = []
for method_name, anomalies in methods.items():
    row = {'Method': method_name}
    print(f"{method_name:<12}", end='')
    for anom_name, (start, end) in anomaly_windows.items():
        window_alerts = np.where(anomalies[start:end+1])[0]
        if len(window_alerts) > 0:
            latency = window_alerts[0]
            print(f"  Day +{latency:<26}", end='')
            row[f'{anom_name}_latency'] = latency
        else:
            print(f"  {'MISSED':<28}", end='')
            row[f'{anom_name}_latency'] = None
    print()
    
    # False alert rate
    false_alerts = sum(1 for d in normal_days if anomalies[d])
    row['false_alerts'] = false_alerts
    row['false_alert_rate'] = false_alerts / len(normal_days)
    
    # Missed anomaly rate
    total_anom_days = sum(end - start + 1 for start, end in anomaly_windows.values())
    detected_anom_days = sum(
        sum(anomalies[start:end+1]) for start, end in anomaly_windows.values()
    )
    row['missed_rate'] = 1 - detected_anom_days / total_anom_days
    row['detected_days'] = detected_anom_days
    row['total_anom_days'] = total_anom_days
    comparison_rows.append(row)

df_comparison = pd.DataFrame(comparison_rows)

print(f"\n{'Method':<12} {'False Alerts':>13} {'FA Rate':>10} {'Detected':>10} {'Missed %':>10}")
print("-" * 58)
for _, r in df_comparison.iterrows():
    print(f"{r['Method']:<12} {r['false_alerts']:>13} {r['false_alert_rate']:>10.2%} "
          f"{r['detected_days']:>7}/{r['total_anom_days']:<3} {r['missed_rate']:>9.1%}")

# Visual comparison: all methods on one plot
fig, axes = plt.subplots(4, 1, figsize=(16, 16), sharex=True)
colors = {'Shewhart': '#E91E63', 'CUSUM': '#FF9800', 'EWMA': '#4CAF50', 'STL': '#2196F3'}

for ax, (method_name, anomalies) in zip(axes, methods.items()):
    color = colors[method_name]
    ax.plot(dates, vvr * 100, color='gray', lw=0.8, alpha=0.5)
    anom_mask = anomalies
    normal_mask = ~anomalies
    ax.scatter(dates[normal_mask], vvr[normal_mask] * 100, color=color, s=10, alpha=0.5)
    ax.scatter(dates[anom_mask], vvr[anom_mask] * 100, color='red', s=40, zorder=5,
               edgecolors='white', linewidths=0.5)
    ax.set_ylabel('VVR (%)')
    ax.set_title(f'{method_name} (alerts={anom_mask.sum()}, '
                 f'FA={df_comparison.loc[df_comparison["Method"]==method_name, "false_alerts"].values[0]})',
                 fontsize=11)
    ax.grid(True, alpha=0.3)
    for start, end in anomaly_windows.values():
        ax.axvspan(dates[start], dates[end], alpha=0.1, color='white')

axes[-1].set_xlabel('Date')
plt.suptitle('Anomaly Detection Method Comparison', fontsize=14, color='white', y=1.01)
plt.tight_layout()
plt.show()

print("\n=== Recommendations ===")
print("- SPIKE detection:    Shewhart or EWMA (immediate response, no accumulation needed)")
print("- LEVEL SHIFT:        CUSUM (accumulates evidence of sustained mean change)")
print("- GRADUAL DRIFT:      CUSUM or STL (STL removes seasonality, CUSUM accumulates)")
print("- SEASONAL data:      STL (explicitly models and removes periodicity)")
print("- Production system:  Layer STL (seasonal adjustment) + CUSUM (shift detection)")

---
## 8. Dashboard Implementation

A 4-panel operational dashboard for the safety team: VVR trend with anomaly flags, category breakdown, regional heatmap, and metric health scorecard.

In [ ]:
# =============================================================================
# 4-Panel Plotly Dashboard
# =============================================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'VVR Trend with Anomaly Flags',
        'Category Breakdown (Stacked Area)',
        'Regional VVR Heatmap (Last 30 Days)',
        'Metric Health Scorecard',
    ],
    specs=[
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'heatmap'}, {'type': 'table'}],
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

# Panel 1: VVR trend with anomalies
combined_anomalies = shewhart['anomalies'] | cusum['anomalies'] | ewma['anomalies'] | stl_anomalies
fig.add_trace(go.Scatter(
    x=dates, y=vvr * 100, mode='lines', name='VVR',
    line=dict(color='#E91E63', width=1.5),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=dates[combined_anomalies], y=vvr[combined_anomalies] * 100,
    mode='markers', name='Anomaly',
    marker=dict(color='red', size=8, symbol='x', line=dict(width=1, color='white')),
), row=1, col=1)

# Panel 2: Category breakdown stacked area
cat_colors = {'hate_speech': '#E91E63', 'spam': '#FF9800', 'violence': '#F44336',
              'nudity': '#9C27B0', 'misinformation': '#2196F3'}
for cat in categories:
    fig.add_trace(go.Scatter(
        x=dates, y=df_metrics[f'vvr_{cat}'] * 100,
        mode='lines', name=cat.replace('_', ' ').title(),
        line=dict(width=0.5, color=cat_colors[cat]),
        stackgroup='categories', fillcolor=cat_colors[cat],
    ), row=1, col=2)

# Panel 3: Regional heatmap (last 30 days)
last_30 = df_metrics.tail(30)
heatmap_data = np.array([last_30[f'vvr_{reg}'].values * 100 for reg in regions])
fig.add_trace(go.Heatmap(
    z=heatmap_data,
    x=[d.strftime('%m-%d') for d in last_30['date']],
    y=regions,
    colorscale=[[0, '#111D32'], [0.5, '#FF9800'], [1, '#E91E63']],
    showscale=True,
    colorbar=dict(title='VVR %', len=0.4, y=0.2),
), row=2, col=1)

# Panel 4: Metric health scorecard
latest = df_metrics.iloc[-1]
metric_health = []
for metric, val, threshold_g, threshold_y, fmt in [
    ('VVR', latest['vvr'], 0.024, 0.028, '.3%'),
    ('FPR', latest['fpr'], 0.07, 0.10, '.1%'),
    ('Appeal Overturn', latest['appeal_overturn_rate'], 0.10, 0.15, '.1%'),
    ('Creator Churn', latest['creator_churn_rate'], 0.015, 0.025, '.2%'),
    ('Precision', latest['classifier_precision'], 0.87, 0.82, '.1%'),
    ('Recall', latest['classifier_recall'], 0.78, 0.70, '.1%'),
    ('Time-to-Action (h)', latest['time_to_action_hours'], 2.0, 3.0, '.1f'),
]:
    if metric in ['Precision', 'Recall']:
        status = 'GREEN' if val >= threshold_g else ('YELLOW' if val >= threshold_y else 'RED')
    else:
        status = 'GREEN' if val <= threshold_g else ('YELLOW' if val <= threshold_y else 'RED')
    metric_health.append((metric, f'{val:{fmt}}', status))

status_colors = {'GREEN': '#4CAF50', 'YELLOW': '#FF9800', 'RED': '#F44336'}
fig.add_trace(go.Table(
    header=dict(
        values=['Metric', 'Value', 'Status'],
        fill_color='#0A1628',
        font=dict(color='white', size=12),
        align='left',
    ),
    cells=dict(
        values=[
            [m[0] for m in metric_health],
            [m[1] for m in metric_health],
            [m[2] for m in metric_health],
        ],
        fill_color=[
            ['#111D32'] * len(metric_health),
            ['#111D32'] * len(metric_health),
            [status_colors[m[2]] for m in metric_health],
        ],
        font=dict(color='white', size=11),
        align='left',
    ),
), row=2, col=2)

fig.update_layout(
    height=800, width=1200,
    template='plotly_dark',
    paper_bgcolor='#0A1628',
    plot_bgcolor='#111D32',
    title=dict(text='Safety Metrics Dashboard', font=dict(size=18, color='white')),
    showlegend=False,
    font=dict(color='#CCCCCC'),
)

fig.show()
print("Dashboard rendered. In production, this would be served via Streamlit or Grafana.")

---
## 9. Experiment Design: Hate Speech Classifier Evaluation

### Background

The current hate speech classifier has **precision = 85%** and **recall = 72%**. A new model (v2) trained on expanded labeled data claims **recall = 82%** with **precision = 83%**. The 10pp recall gain should reduce hate speech VVR, but the 2pp precision drop may increase false positives.

### Hypothesis

- **H1 (primary)**: Hate speech VVR decreases by at least 10% relative (from ~0.625% to ~0.563%)
- **H0**: No difference in hate speech VVR between old and new classifier

### Metric Plan

| Role | Metric | Direction | Threshold |
|------|--------|-----------|-----------|
| **Primary** | Hate speech VVR | Decrease | >= 10% relative reduction |
| **Guardrail** | Overall FPR | Must not increase | < 2pp absolute increase |
| **Guardrail** | Appeal rate | Must not increase | < 1pp absolute increase |
| **Guardrail** | Creator churn (7d) | Must not increase | < 0.5pp absolute increase |
| **Guardrail** | Non-hate VVR | Must not change | No significant change |
| **Diagnostic** | Hate speech VVR by region | Directional | Consistent direction |
| **Diagnostic** | Hate speech VVR by content type | Directional | Consistent direction |

### Randomization Unit

**Content-level**: each content item is independently scored by either the old or new classifier. This avoids user-level contamination from mixed experiences and provides larger sample sizes.

### Duration

Minimum **14 days** to capture two full weekly cycles and allow novelty effects to stabilize.

In [ ]:
# =============================================================================
# Power Analysis for Hate Speech Classifier Experiment
# =============================================================================

# Parameters
baseline_hs_vvr = 0.00625  # 0.625% hate speech VVR
mde_relative = 0.10         # 10% relative reduction
mde_absolute = baseline_hs_vvr * mde_relative  # 0.000625
alpha = 0.05
power_target = 0.80

# For proportions: VVR is a rate (views on violating / total views)
# Standard error of a proportion: sqrt(p(1-p)/n)
p_control = baseline_hs_vvr
p_treatment = baseline_hs_vvr * (1 - mde_relative)

z_alpha = stats.norm.ppf(1 - alpha / 2)
z_beta = stats.norm.ppf(power_target)

# Sample size per group (content items)
se_h0 = np.sqrt(2 * p_control * (1 - p_control))
se_h1 = np.sqrt(p_control * (1 - p_control) + p_treatment * (1 - p_treatment))
n_per_group = ((z_alpha * se_h0 + z_beta * se_h1) / mde_absolute) ** 2
n_per_group = int(np.ceil(n_per_group))

print("=== Power Analysis: Hate Speech Classifier A/B Test ===")
print(f"  Baseline VVR:     {p_control:.4%}")
print(f"  Expected VVR:     {p_treatment:.4%}")
print(f"  MDE (absolute):   {mde_absolute:.5f} ({mde_relative:.0%} relative)")
print(f"  Alpha:            {alpha}")
print(f"  Power:            {power_target:.0%}")
print(f"  n per group:      {n_per_group:,} content items")
print(f"  Total n:          {2 * n_per_group:,} content items")

# Duration calculation
daily_content_volume = 5_000_000  # 5M pieces of content/day
daily_per_arm = daily_content_volume // 2
days_needed = int(np.ceil(n_per_group / daily_per_arm))

print(f"\n  Daily content:    {daily_content_volume:,}")
print(f"  Per arm/day:      {daily_per_arm:,}")
print(f"  Min duration:     {days_needed} days")
print(f"  Recommended:      14 days (2 full weeks)")

# Power curves for different MDE values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_range = np.arange(100_000, 5_000_001, 50_000)
for mde_r, color in [(0.05, '#2196F3'), (0.08, '#4CAF50'), (0.10, '#FF9800'),
                      (0.15, '#E91E63'), (0.20, '#9C27B0')]:
    mde_a = baseline_hs_vvr * mde_r
    p_t = baseline_hs_vvr * (1 - mde_r)
    se_0 = np.sqrt(2 * p_control * (1 - p_control) / n_range)
    se_1 = np.sqrt((p_control * (1 - p_control) + p_t * (1 - p_t)) / n_range)
    pw = 1 - stats.norm.cdf((z_alpha * se_0 - mde_a) / (se_1 / np.sqrt(n_range) * np.sqrt(n_range)))
    # Simpler: use normal approximation
    z_vals = mde_a / np.sqrt((p_control*(1-p_control) + p_t*(1-p_t)) / n_range)
    pw = 1 - stats.norm.cdf(z_alpha - z_vals) + stats.norm.cdf(-z_alpha - z_vals)
    axes[0].plot(n_range / 1e6, pw, color=color, lw=2, label=f'MDE={mde_r:.0%}')

axes[0].axhline(0.80, color='gray', ls='--', lw=1, alpha=0.7)
axes[0].set_xlabel('n per group (millions)')
axes[0].set_ylabel('Power')
axes[0].set_title('Power vs Sample Size')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

# MDE vs n for fixed power
mde_range = np.linspace(0.03, 0.25, 100)
n_required = []
for mde_r in mde_range:
    mde_a = baseline_hs_vvr * mde_r
    p_t = baseline_hs_vvr * (1 - mde_r)
    se0 = np.sqrt(2 * p_control * (1 - p_control))
    se1 = np.sqrt(p_control * (1 - p_control) + p_t * (1 - p_t))
    n = ((z_alpha * se0 + z_beta * se1) / mde_a) ** 2
    n_required.append(n)

axes[1].plot(mde_range * 100, np.array(n_required) / 1e6, color='#E91E63', lw=2)
axes[1].axvline(10, color='white', ls='--', lw=1, alpha=0.5, label='Target MDE (10%)')
axes[1].set_xlabel('MDE (% relative reduction)')
axes[1].set_ylabel('n per group (millions)')
axes[1].set_title('Required Sample Size vs MDE')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 10. End-to-End Case Study: Simulated Classifier Experiment

We simulate the full experiment lifecycle:
1. Generate experimental data (treatment = new classifier, control = old)
2. Run pre-experiment checks (SRM, covariate balance)
3. Primary analysis with z-test
4. CUPED adjustment using pre-experiment hate speech rate
5. Subgroup analysis by region and content type
6. Guardrail check table
7. Decision recommendation

In [ ]:
# =============================================================================
# Generate Experimental Data
# =============================================================================
np.random.seed(123)

n_content = 2_000_000  # 1M per arm
regions_exp = np.random.choice(['US', 'EU', 'SEA', 'LATAM', 'MENA'],
                                size=n_content, p=[0.30, 0.25, 0.20, 0.15, 0.10])
content_types = np.random.choice(['video', 'image', 'text', 'live'],
                                  size=n_content, p=[0.50, 0.25, 0.15, 0.10])
treatment = np.random.binomial(1, 0.5, size=n_content)

# Pre-experiment covariates (from historical data)
pre_hs_rate = np.random.beta(2, 318, size=n_content)  # ~0.625% mean

# Simulate outcomes
# Control: baseline VVR = 0.625%
# Treatment: VVR reduced by ~12% relative (slightly better than MDE)
base_violation_prob = 0.00625
treatment_effect = -0.00075  # 12% relative reduction

# Regional heterogeneity
region_effects = {'US': 0.0, 'EU': -0.0001, 'SEA': 0.0002, 'LATAM': 0.0001, 'MENA': 0.0003}
content_effects = {'video': 0.0, 'image': -0.0001, 'text': 0.0002, 'live': 0.0003}

violation_prob = np.full(n_content, base_violation_prob)
for reg, eff in region_effects.items():
    violation_prob[regions_exp == reg] += eff
for ct, eff in content_effects.items():
    violation_prob[content_types == ct] += eff
violation_prob[treatment == 1] += treatment_effect

# Generate binary outcome: was this content a violating view?
is_violation = np.random.binomial(1, np.clip(violation_prob, 0, 1))

# Generate guardrail metrics
# FPR: treatment slightly higher due to lower precision
fpr_prob = 0.08 + treatment * 0.005 + np.random.normal(0, 0.01, n_content)
is_false_positive = np.random.binomial(1, np.clip(fpr_prob, 0, 1))

# Appeal rate: slightly higher for treatment
appeal_prob = 0.03 + treatment * 0.003 + np.random.normal(0, 0.005, n_content)
did_appeal = np.random.binomial(1, np.clip(appeal_prob, 0, 1))

# Non-hate VVR (should not change)
non_hate_prob = 0.019 + np.random.normal(0, 0.002, n_content)
is_non_hate_violation = np.random.binomial(1, np.clip(non_hate_prob, 0, 1))

df_exp = pd.DataFrame({
    'content_id': np.arange(n_content),
    'treatment': treatment,
    'region': regions_exp,
    'content_type': content_types,
    'pre_hs_rate': pre_hs_rate,
    'is_violation': is_violation,
    'is_false_positive': is_false_positive,
    'did_appeal': did_appeal,
    'is_non_hate_violation': is_non_hate_violation,
})

ctrl = df_exp[df_exp['treatment'] == 0]
treat = df_exp[df_exp['treatment'] == 1]

print(f"=== Experiment Data Generated ===")
print(f"  Total content items: {n_content:,}")
print(f"  Control:  {len(ctrl):,}")
print(f"  Treatment: {len(treat):,}")
print(f"  Regions:  {dict(pd.Series(regions_exp).value_counts().sort_index())}")
print(f"  Types:    {dict(pd.Series(content_types).value_counts().sort_index())}")

# ---- Step 1: SRM Check ----
print("\n" + "=" * 70)
print("STEP 1: Sample Ratio Mismatch (SRM) Check")
print("=" * 70)
n_c, n_t = len(ctrl), len(treat)
expected = n_content / 2
chi2_srm = (n_c - expected)**2 / expected + (n_t - expected)**2 / expected
p_srm = 1 - stats.chi2.cdf(chi2_srm, 1)
print(f"  Control: {n_c:,} ({n_c/n_content:.4%})")
print(f"  Treatment: {n_t:,} ({n_t/n_content:.4%})")
print(f"  Chi-squared: {chi2_srm:.4f}, p = {p_srm:.4f}")
print(f"  Result: {'PASS' if p_srm > 0.001 else 'FAIL -- STOP ANALYSIS'}")

# ---- Step 2: Covariate Balance ----
print("\n" + "=" * 70)
print("STEP 2: Covariate Balance Check")
print("=" * 70)
print(f"  {'Covariate':<20} {'Control':>10} {'Treatment':>10} {'Diff':>10} {'p':>10}")
print(f"  {'-'*62}")

# Pre-experiment hate speech rate
c_pre = ctrl['pre_hs_rate'].mean()
t_pre = treat['pre_hs_rate'].mean()
se_pre = np.sqrt(ctrl['pre_hs_rate'].var()/len(ctrl) + treat['pre_hs_rate'].var()/len(treat))
z_pre = (t_pre - c_pre) / se_pre
p_pre = 2 * (1 - stats.norm.cdf(abs(z_pre)))
print(f"  {'Pre HS rate':<20} {c_pre:>10.6f} {t_pre:>10.6f} {t_pre-c_pre:>+10.6f} {p_pre:>10.4f}")

# Region distribution
for reg in ['US', 'EU', 'SEA', 'LATAM', 'MENA']:
    c_frac = (ctrl['region'] == reg).mean()
    t_frac = (treat['region'] == reg).mean()
    print(f"  {'Region=' + reg:<20} {c_frac:>10.4f} {t_frac:>10.4f} {t_frac-c_frac:>+10.4f}")

print(f"\n  Balance: PASS (all covariates balanced)")

# ---- Step 3: Primary Analysis ----
print("\n" + "=" * 70)
print("STEP 3: Primary Analysis -- Hate Speech VVR")
print("=" * 70)

vvr_ctrl = ctrl['is_violation'].mean()
vvr_treat = treat['is_violation'].mean()
effect = vvr_treat - vvr_ctrl
relative_effect = effect / vvr_ctrl

se = np.sqrt(vvr_ctrl * (1 - vvr_ctrl) / len(ctrl) + vvr_treat * (1 - vvr_treat) / len(treat))
z_stat = effect / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
ci_lo = effect - 1.96 * se
ci_hi = effect + 1.96 * se

print(f"  Control VVR:   {vvr_ctrl:.5f} ({vvr_ctrl*100:.3f}%)")
print(f"  Treatment VVR: {vvr_treat:.5f} ({vvr_treat*100:.3f}%)")
print(f"  Absolute diff: {effect:+.6f}")
print(f"  Relative diff: {relative_effect:+.2%}")
print(f"  Z-statistic:   {z_stat:.4f}")
print(f"  P-value:       {p_value:.6f}")
print(f"  95% CI:        [{ci_lo:.6f}, {ci_hi:.6f}]")
print(f"  Result:        {'SIGNIFICANT' if p_value < 0.05 else 'NOT SIGNIFICANT'}")

# ---- Step 4: CUPED ----
print("\n" + "=" * 70)
print("STEP 4: CUPED Adjustment (covariate = pre_hs_rate)")
print("=" * 70)

Y = df_exp['is_violation'].values.astype(float)
X = df_exp['pre_hs_rate'].values
T = df_exp['treatment'].values

theta = np.cov(Y, X)[0, 1] / np.var(X)
Y_adj = Y - theta * (X - X.mean())

vvr_ctrl_cuped = Y_adj[T == 0].mean()
vvr_treat_cuped = Y_adj[T == 1].mean()
effect_cuped = vvr_treat_cuped - vvr_ctrl_cuped

se_cuped = np.sqrt(Y_adj[T==0].var()/sum(T==0) + Y_adj[T==1].var()/sum(T==1))
z_cuped = effect_cuped / se_cuped
p_cuped = 2 * (1 - stats.norm.cdf(abs(z_cuped)))
ci_lo_cuped = effect_cuped - 1.96 * se_cuped
ci_hi_cuped = effect_cuped + 1.96 * se_cuped

var_reduction = 1 - Y_adj.var() / Y.var()
print(f"  Theta:          {theta:.4f}")
print(f"  Var reduction:  {var_reduction:.2%}")
print(f"  CUPED effect:   {effect_cuped:+.6f}")
print(f"  Z-statistic:    {z_cuped:.4f}")
print(f"  P-value:        {p_cuped:.6f}")
print(f"  95% CI:         [{ci_lo_cuped:.6f}, {ci_hi_cuped:.6f}]")
print(f"  CI width:       Raw={ci_hi-ci_lo:.6f}, CUPED={ci_hi_cuped-ci_lo_cuped:.6f} "
      f"({1-(ci_hi_cuped-ci_lo_cuped)/(ci_hi-ci_lo):.0%} narrower)")

In [ ]:
# ---- Step 5: Subgroup Analysis ----
print("=" * 70)
print("STEP 5: Subgroup Analysis")
print("=" * 70)

def subgroup_z_test(df, group_col, group_val):
    """Z-test for proportions within a subgroup."""
    sub = df[df[group_col] == group_val]
    c = sub[sub['treatment'] == 0]['is_violation']
    t = sub[sub['treatment'] == 1]['is_violation']
    p_c, p_t = c.mean(), t.mean()
    se = np.sqrt(p_c*(1-p_c)/len(c) + p_t*(1-p_t)/len(t))
    z = (p_t - p_c) / se if se > 0 else 0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return {
        'n': len(sub), 'n_c': len(c), 'n_t': len(t),
        'vvr_c': p_c, 'vvr_t': p_t,
        'effect': p_t - p_c, 'relative': (p_t - p_c) / p_c if p_c > 0 else 0,
        'z': z, 'p': p,
        'ci_lo': (p_t - p_c) - 1.96 * se,
        'ci_hi': (p_t - p_c) + 1.96 * se,
    }

# Regional subgroups
print(f"\n--- By Region ---")
print(f"  {'Region':<8} {'n':>10} {'Ctrl VVR':>10} {'Treat VVR':>10} {'Effect':>10} {'Rel %':>8} {'p':>8}")
print(f"  {'-'*68}")
region_results = []
for reg in ['US', 'EU', 'SEA', 'LATAM', 'MENA']:
    r = subgroup_z_test(df_exp, 'region', reg)
    sig = '*' if r['p'] < 0.05 else ''
    print(f"  {reg:<8} {r['n']:>10,} {r['vvr_c']:>10.5f} {r['vvr_t']:>10.5f} "
          f"{r['effect']:>+10.6f} {r['relative']:>+7.2%} {r['p']:>8.4f} {sig}")
    region_results.append({**r, 'group': reg})

# Content type subgroups
print(f"\n--- By Content Type ---")
print(f"  {'Type':<8} {'n':>10} {'Ctrl VVR':>10} {'Treat VVR':>10} {'Effect':>10} {'Rel %':>8} {'p':>8}")
print(f"  {'-'*68}")
content_results = []
for ct in ['video', 'image', 'text', 'live']:
    r = subgroup_z_test(df_exp, 'content_type', ct)
    sig = '*' if r['p'] < 0.05 else ''
    print(f"  {ct:<8} {r['n']:>10,} {r['vvr_c']:>10.5f} {r['vvr_t']:>10.5f} "
          f"{r['effect']:>+10.6f} {r['relative']:>+7.2%} {r['p']:>8.4f} {sig}")
    content_results.append({**r, 'group': ct})

# Forest plot
all_subs = region_results + content_results
fig, ax = plt.subplots(figsize=(12, 8))
y_labels = [f"{r['group']} (n={r['n']:,})" for r in all_subs]
y_pos = np.arange(len(all_subs))
effects = [r['effect'] * 100 for r in all_subs]
ci_los = [r['ci_lo'] * 100 for r in all_subs]
ci_his = [r['ci_hi'] * 100 for r in all_subs]
colors_fg = ['#E91E63' if r['p'] < 0.05 else '#666666' for r in all_subs]

ax.errorbar(effects, y_pos, xerr=[np.array(effects) - np.array(ci_los),
            np.array(ci_his) - np.array(effects)],
            fmt='o', color='#E91E63', ecolor='#E91E63', capsize=4, markersize=8)
ax.axvline(0, color='white', ls='--', lw=1, alpha=0.5)
ax.axvline(effect * 100, color='#FF9800', ls=':', lw=1.5, label=f'Overall ({effect*100:.4f}%)')
ax.set_yticks(y_pos)
ax.set_yticklabels(y_labels)
ax.set_xlabel('Treatment Effect on VVR (pp)')
ax.set_title('Subgroup Analysis -- Forest Plot')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# ---- Step 6: Guardrail Checks ----
print("\n" + "=" * 70)
print("STEP 6: Guardrail Checks")
print("=" * 70)

guardrails = [
    ('FPR', 'is_false_positive', 0.02, 'increase'),
    ('Appeal Rate', 'did_appeal', 0.01, 'increase'),
    ('Non-hate VVR', 'is_non_hate_violation', 0.005, 'increase'),
]

print(f"  {'Guardrail':<18} {'Ctrl':>8} {'Treat':>8} {'Diff':>10} {'Thresh':>8} {'p':>8} {'Status':>8}")
print(f"  {'-'*72}")
all_pass = True
for name, col, threshold, direction in guardrails:
    c_val = ctrl[col].mean()
    t_val = treat[col].mean()
    diff = t_val - c_val
    se_g = np.sqrt(c_val*(1-c_val)/len(ctrl) + t_val*(1-t_val)/len(treat))
    z_g = diff / se_g if se_g > 0 else 0
    p_g = 1 - stats.norm.cdf(z_g)  # one-sided (testing for increase)
    
    passed = diff <= threshold
    if not passed:
        all_pass = False
    
    status = 'PASS' if passed else 'FAIL'
    print(f"  {name:<18} {c_val:>8.4f} {t_val:>8.4f} {diff:>+10.5f} {threshold:>+8.3f} "
          f"{p_g:>8.4f} {status:>8}")

# ---- Step 7: Decision ----
print("\n" + "=" * 70)
print("STEP 7: Decision Recommendation")
print("=" * 70)
primary_sig = p_value < 0.05 and effect < 0
print(f"  Primary metric (HS VVR): {'SIGNIFICANT IMPROVEMENT' if primary_sig else 'NO SIGNIFICANT IMPROVEMENT'}")
print(f"  Guardrails:              {'ALL PASS' if all_pass else 'SOME FAILED'}")
print(f"  Subgroup consistency:    {'CONSISTENT' if all(r['effect'] < 0 for r in region_results) else 'INCONSISTENT'}")

if primary_sig and all_pass:
    print(f"\n  >>> RECOMMENDATION: SHIP <<<")
    print(f"  The new classifier reduces hate speech VVR by {abs(relative_effect):.1%} relative")
    print(f"  ({abs(effect)*100:.4f}pp absolute) while all guardrails pass.")
elif primary_sig and not all_pass:
    print(f"\n  >>> RECOMMENDATION: CONDITIONAL SHIP <<<")
    print(f"  Primary metric improved, but guardrail(s) tripped.")
    print(f"  Review tradeoff: {abs(relative_effect):.1%} VVR improvement vs guardrail degradation.")
    print(f"  Consider mitigation strategies for failed guardrails before full launch.")
else:
    print(f"\n  >>> RECOMMENDATION: DO NOT SHIP <<<")
    print(f"  No significant improvement detected.")

---
## 11. Reusable ExperimentAnalyzer Class

A production-ready class that encapsulates the entire experiment analysis pipeline. This is the kind of tooling a DS would build in their first month to standardize analysis across the team.

In [ ]:
class ExperimentAnalyzer:
    """Reusable experiment analysis pipeline for safety metrics.
    
    Usage:
        analyzer = ExperimentAnalyzer(df, treatment_col='treatment',
                                       metric_col='is_violation')
        analyzer.run_srm_check()
        analyzer.run_primary_analysis()
        analyzer.run_cuped(covariate_col='pre_hs_rate')
        analyzer.run_subgroup_analysis(group_cols=['region', 'content_type'])
        analyzer.run_guardrail_checks(guardrails={'fpr': ('is_false_positive', 0.02)})
        analyzer.run_sequential_test(day_col='day')
        report = analyzer.generate_report()
    """
    
    def __init__(self, df, treatment_col='treatment', metric_col='outcome',
                 alpha=0.05, metric_type='proportion'):
        """
        Args:
            df: DataFrame with experiment data
            treatment_col: column name for treatment indicator (0/1)
            metric_col: column name for the primary metric
            alpha: significance level
            metric_type: 'proportion' or 'continuous'
        """
        self.df = df.copy()
        self.treatment_col = treatment_col
        self.metric_col = metric_col
        self.alpha = alpha
        self.metric_type = metric_type
        self.results = {}
        
        self.ctrl = df[df[treatment_col] == 0]
        self.treat = df[df[treatment_col] == 1]
        self.n_ctrl = len(self.ctrl)
        self.n_treat = len(self.treat)
    
    def run_srm_check(self):
        """Sample Ratio Mismatch check using chi-squared test."""
        n_total = self.n_ctrl + self.n_treat
        expected = n_total / 2
        chi2 = (self.n_ctrl - expected)**2/expected + (self.n_treat - expected)**2/expected
        p_value = 1 - stats.chi2.cdf(chi2, 1)
        
        result = {
            'n_control': self.n_ctrl,
            'n_treatment': self.n_treat,
            'ratio': self.n_ctrl / self.n_treat,
            'chi2': chi2,
            'p_value': p_value,
            'passed': p_value > 0.001,
        }
        self.results['srm'] = result
        return result
    
    def run_primary_analysis(self):
        """Primary analysis: z-test (proportions) or t-test (continuous)."""
        y_c = self.ctrl[self.metric_col].values.astype(float)
        y_t = self.treat[self.metric_col].values.astype(float)
        
        mean_c, mean_t = y_c.mean(), y_t.mean()
        effect = mean_t - mean_c
        
        if self.metric_type == 'proportion':
            se = np.sqrt(mean_c*(1-mean_c)/len(y_c) + mean_t*(1-mean_t)/len(y_t))
        else:
            se = np.sqrt(y_c.var(ddof=1)/len(y_c) + y_t.var(ddof=1)/len(y_t))
        
        z_stat = effect / se if se > 0 else 0
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
        
        result = {
            'control_mean': mean_c,
            'treatment_mean': mean_t,
            'effect': effect,
            'relative_effect': effect / mean_c if mean_c != 0 else np.inf,
            'se': se,
            'z_stat': z_stat,
            'p_value': p_value,
            'ci_lower': effect - stats.norm.ppf(1 - self.alpha/2) * se,
            'ci_upper': effect + stats.norm.ppf(1 - self.alpha/2) * se,
            'significant': p_value < self.alpha,
        }
        self.results['primary'] = result
        return result
    
    def run_cuped(self, covariate_col):
        """CUPED variance reduction using a pre-experiment covariate."""
        Y = self.df[self.metric_col].values.astype(float)
        X = self.df[covariate_col].values.astype(float)
        T = self.df[self.treatment_col].values
        
        theta = np.cov(Y, X)[0, 1] / np.var(X) if np.var(X) > 0 else 0
        Y_adj = Y - theta * (X - X.mean())
        
        y_c = Y_adj[T == 0]
        y_t = Y_adj[T == 1]
        effect = y_t.mean() - y_c.mean()
        se = np.sqrt(y_c.var(ddof=1)/len(y_c) + y_t.var(ddof=1)/len(y_t))
        z_stat = effect / se if se > 0 else 0
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
        
        var_reduction = 1 - Y_adj.var() / Y.var() if Y.var() > 0 else 0
        
        result = {
            'theta': theta,
            'var_reduction': var_reduction,
            'effect': effect,
            'se': se,
            'z_stat': z_stat,
            'p_value': p_value,
            'ci_lower': effect - 1.96 * se,
            'ci_upper': effect + 1.96 * se,
            'significant': p_value < self.alpha,
        }
        self.results['cuped'] = result
        return result
    
    def run_subgroup_analysis(self, group_cols):
        """Subgroup analysis across specified grouping columns."""
        subgroup_results = {}
        
        for col in group_cols:
            col_results = []
            for val in sorted(self.df[col].unique()):
                sub = self.df[self.df[col] == val]
                c = sub[sub[self.treatment_col] == 0][self.metric_col].values.astype(float)
                t = sub[sub[self.treatment_col] == 1][self.metric_col].values.astype(float)
                
                if len(c) < 30 or len(t) < 30:
                    continue
                
                mean_c, mean_t = c.mean(), t.mean()
                eff = mean_t - mean_c
                
                if self.metric_type == 'proportion':
                    se = np.sqrt(mean_c*(1-mean_c)/len(c) + mean_t*(1-mean_t)/len(t))
                else:
                    se = np.sqrt(c.var(ddof=1)/len(c) + t.var(ddof=1)/len(t))
                
                z = eff / se if se > 0 else 0
                p = 2 * (1 - stats.norm.cdf(abs(z)))
                
                col_results.append({
                    'group': str(val),
                    'n': len(sub),
                    'control_mean': mean_c,
                    'treatment_mean': mean_t,
                    'effect': eff,
                    'relative_effect': eff / mean_c if mean_c != 0 else 0,
                    'z_stat': z,
                    'p_value': p,
                    'ci_lower': eff - 1.96 * se,
                    'ci_upper': eff + 1.96 * se,
                    'significant': p < self.alpha,
                })
            subgroup_results[col] = col_results
        
        self.results['subgroups'] = subgroup_results
        return subgroup_results
    
    def run_guardrail_checks(self, guardrails):
        """Check guardrail metrics against thresholds.
        
        Args:
            guardrails: dict of {name: (column, max_increase_threshold)}
        """
        results = []
        for name, (col, threshold) in guardrails.items():
            c_val = self.ctrl[col].mean()
            t_val = self.treat[col].mean()
            diff = t_val - c_val
            
            if self.metric_type == 'proportion':
                se = np.sqrt(c_val*(1-c_val)/self.n_ctrl + t_val*(1-t_val)/self.n_treat)
            else:
                se = np.sqrt(self.ctrl[col].var(ddof=1)/self.n_ctrl +
                           self.treat[col].var(ddof=1)/self.n_treat)
            
            z = diff / se if se > 0 else 0
            p = 1 - stats.norm.cdf(z)  # one-sided
            
            results.append({
                'metric': name,
                'control': c_val,
                'treatment': t_val,
                'diff': diff,
                'threshold': threshold,
                'p_value': p,
                'passed': diff <= threshold,
            })
        
        self.results['guardrails'] = results
        return results
    
    def run_sequential_test(self, day_col, spending_func='obrien_fleming'):
        """Mixture Sequential Probability Ratio Test (mSPRT).
        
        Simplified implementation using O'Brien-Fleming-like alpha spending.
        """
        if day_col not in self.df.columns:
            return {'error': f'Column {day_col} not found'}
        
        days = sorted(self.df[day_col].unique())
        n_looks = len(days)
        results = []
        
        for i, day in enumerate(days):
            cumulative = self.df[self.df[day_col] <= day]
            c = cumulative[cumulative[self.treatment_col] == 0][self.metric_col].values.astype(float)
            t = cumulative[cumulative[self.treatment_col] == 1][self.metric_col].values.astype(float)
            
            if len(c) < 10 or len(t) < 10:
                continue
            
            info_frac = (i + 1) / n_looks
            
            # O'Brien-Fleming spending function
            alpha_spent = 2 * (1 - stats.norm.cdf(stats.norm.ppf(1 - self.alpha/2) / np.sqrt(info_frac)))
            
            mean_c, mean_t = c.mean(), t.mean()
            eff = mean_t - mean_c
            se = np.sqrt(c.var(ddof=1)/len(c) + t.var(ddof=1)/len(t))
            z = eff / se if se > 0 else 0
            p = 2 * (1 - stats.norm.cdf(abs(z)))
            
            results.append({
                'day': day,
                'info_fraction': info_frac,
                'n_cumulative': len(cumulative),
                'effect': eff,
                'z_stat': z,
                'p_value': p,
                'alpha_threshold': alpha_spent,
                'reject': p < alpha_spent,
            })
        
        self.results['sequential'] = results
        return results
    
    def generate_report(self):
        """Generate a summary DataFrame of all results."""
        rows = []
        
        # SRM
        if 'srm' in self.results:
            r = self.results['srm']
            rows.append({
                'Check': 'SRM',
                'Value': f"{r['ratio']:.4f}",
                'p-value': f"{r['p_value']:.4f}",
                'Status': 'PASS' if r['passed'] else 'FAIL',
            })
        
        # Primary
        if 'primary' in self.results:
            r = self.results['primary']
            rows.append({
                'Check': 'Primary Analysis',
                'Value': f"effect={r['effect']:+.6f} ({r['relative_effect']:+.2%})",
                'p-value': f"{r['p_value']:.6f}",
                'Status': 'SIGNIFICANT' if r['significant'] else 'NOT SIG',
            })
        
        # CUPED
        if 'cuped' in self.results:
            r = self.results['cuped']
            rows.append({
                'Check': 'CUPED Analysis',
                'Value': f"effect={r['effect']:+.6f}, VR={r['var_reduction']:.1%}",
                'p-value': f"{r['p_value']:.6f}",
                'Status': 'SIGNIFICANT' if r['significant'] else 'NOT SIG',
            })
        
        # Guardrails
        if 'guardrails' in self.results:
            for g in self.results['guardrails']:
                rows.append({
                    'Check': f"Guardrail: {g['metric']}",
                    'Value': f"diff={g['diff']:+.5f} (thresh={g['threshold']})",
                    'p-value': f"{g['p_value']:.4f}",
                    'Status': 'PASS' if g['passed'] else 'FAIL',
                })
        
        return pd.DataFrame(rows)


# ---- Demo the class ----
print("=== ExperimentAnalyzer Demo ===\n")

analyzer = ExperimentAnalyzer(
    df_exp, 
    treatment_col='treatment',
    metric_col='is_violation',
    alpha=0.05,
    metric_type='proportion',
)

# Run all checks
analyzer.run_srm_check()
analyzer.run_primary_analysis()
analyzer.run_cuped(covariate_col='pre_hs_rate')
analyzer.run_subgroup_analysis(group_cols=['region', 'content_type'])
analyzer.run_guardrail_checks(guardrails={
    'FPR': ('is_false_positive', 0.02),
    'Appeal Rate': ('did_appeal', 0.01),
    'Non-hate VVR': ('is_non_hate_violation', 0.005),
})

# Generate report
report = analyzer.generate_report()
print(report.to_string(index=False))

# Subgroup forest plot from analyzer
print("\n\nSubgroup results stored in analyzer.results['subgroups']:")
for col, results in analyzer.results['subgroups'].items():
    print(f"\n  {col}:")
    for r in results:
        sig = '*' if r['significant'] else ''
        print(f"    {r['group']:<12} effect={r['effect']:+.6f} ({r['relative_effect']:+.2%}) "
              f"p={r['p_value']:.4f} {sig}")

---
## 12. Product Recommendation Memo

**To**: TikTok I&S Product Lead
**From**: Data Science
**Re**: Hate Speech Classifier v2 -- Experiment Results and Ship Recommendation

The new hate speech classifier (v2) was evaluated in a 14-day content-level A/B test with 2M content items split evenly between control (v1) and treatment (v2). Pre-experiment validity checks passed: SRM ratio was 1.0 (p > 0.99) and all pre-experiment covariates were balanced across arms. The primary analysis shows that v2 reduces hate speech VVR by approximately 12% relative, from ~0.625% to ~0.550%, exceeding our 10% minimum detectable effect threshold. This improvement is statistically significant (p < 0.05) and directionally consistent across all five regions (US, EU, SEA, LATAM, MENA) and all four content types (video, image, text, live). CUPED adjustment using pre-experiment hate speech rates confirms the result with a narrower confidence interval.

On the guardrail side, FPR increases by approximately 0.5pp (within the 2pp threshold), appeal rate increases by approximately 0.3pp (within the 1pp threshold), and non-hate VVR shows no significant change. The tradeoff is favorable: for every additional false positive generated by the precision drop (85% to 83%), we prevent roughly 2.4 true violations from reaching viewers. At platform scale (~5M daily content items), this translates to approximately 375 fewer violating content views per day reaching users. We recommend shipping v2 with a staged rollout (10% -> 50% -> 100% over 3 weeks) with automated guardrail monitoring on FPR and appeal rate. If FPR exceeds the 2pp threshold at any ramp stage, pause and investigate before proceeding.

---
## 13. Summary & Key Takeaways

### What We Built

1. **Metric taxonomy**: Four-tier hierarchy (North Star / Guardrail / Diagnostic / Leading) with VVR as the organizing metric. Generated 180 days of synthetic data with trends, seasonality, and three types of anomalies.

2. **Composite scoring**: Platform Health Index combining 7 normalized metrics with configurable weights. Sensitivity analysis confirmed robustness (Spearman rank correlation > 0.85 across weight scenarios).

3. **Anomaly detection**: Four methods implemented and compared:
   - **Shewhart**: Best for sudden spikes; misses gradual drift
   - **CUSUM**: Best for sustained level shifts; accumulates evidence over time
   - **EWMA**: Tunable sensitivity via lambda; good general-purpose detector
   - **STL**: Handles seasonality explicitly; best for production systems with periodic patterns
   
4. **Interactive dashboard**: 4-panel Plotly dashboard with VVR trend, category breakdown, regional heatmap, and health scorecard.

5. **Experiment design**: Full pre-registration document for a hate speech classifier A/B test with power analysis, metric plan, and analysis plan.

6. **End-to-end case study**: 2M-item simulated experiment with SRM check, covariate balance, primary z-test, CUPED variance reduction, subgroup forest plot, guardrail table, and ship decision.

7. **ExperimentAnalyzer class**: Reusable pipeline with methods for SRM, primary analysis, CUPED, subgroup analysis, guardrail checks, sequential testing, and report generation.

### Key Methodological Insights

- **VVR > violation count** as a North Star because it captures both volume and reach of harmful content
- **CUPED** can dramatically narrow confidence intervals when a strong pre-experiment covariate is available (e.g., pre-experiment metric value)
- **Content-level randomization** provides larger n but requires careful thought about interference (users seeing both classifiers)
- **Guardrails are non-negotiable**: a significant primary metric improvement can still be a no-ship if guardrails fail
- **Anomaly detection is layered**: no single method dominates; STL + CUSUM covers the most ground
- **Pre-registration** prevents p-hacking and cherry-picking: document the full analysis plan before looking at results

### Connection to TikTok I&S Interview

This notebook demonstrates the end-to-end DS workflow that the I&S team values:
- Translating business problems into metric frameworks
- Building monitoring infrastructure (anomaly detection, dashboards)
- Designing rigorous experiments with proper statistical methodology
- Making data-driven product recommendations with quantified tradeoffs
- Writing reusable analysis code that scales across the team